## OpenStreetMap-Hinweise

Die Analyse nutzt Photon (OSM-Geocoding) fuer die unten angegebenen Eishallen und OSRM fuer geschaetzte Auto-Fahrzeiten. Der oeffentliche Photon-Dienst darf nur in vernuenftigem Umfang genutzt werden; intensive Nutzung kann gedrosselt oder gesperrt werden. Koordinaten und Routen werden lokal gespeichert und wiederverwendet. Die Fahrzeiten sind keine matchtagsbezogenen Echtzeitwerte. Datenquelle/Attribution: © OpenStreetMap contributors (https://www.openstreetmap.org/copyright). Photon-Nutzungsbedingungen: https://github.com/komoot/photon#demo-server. OSRM-Tabellen-API: https://project-osrm.org/docs/v5.24.0/api/#table-service. Geocoding-Ergebnisse bitte anhand der ausgegebenen Hallennamen pruefen; falls keine Halle, sondern z.B. eine Haltestelle gefunden wurde, Koordinaten im Cache korrigieren.

In [1]:
import hashlib
import json
import time
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import statsmodels.api as sm

# Alle gespeicherten Saison-CSV-Dateien laden.
dateien = sorted(Path('csv_data').glob('nl_*_results.csv'))
if not dateien:
    raise FileNotFoundError('Keine Saison-CSV-Dateien in csv_data gefunden.')
df = pd.concat(
    [pd.read_csv(pfad, sep=';', encoding='utf-8-sig').assign(
        Saison=pfad.stem[3:-8].replace('_', '/')) for pfad in dateien],
    ignore_index=True,
)

# Konkrete Eisbahnen statt Teamnamen geocodieren.
arena_suchen = {
    'Ajoie': 'Patinoire Ajoie Porrentruy',
    'Ambri-Piotta': 'Gottardo Arena Quinto',
    'Bern': 'PostFinance Arena Bern',
    'Biel': 'Tissot Arena Biel',
    'Davos': 'Eisstadion Davos',
    'EHC Kloten': 'SWISS Arena Kloten',
    'Fribourg': 'BCF Arena Fribourg',
    'Langnau Tigers': 'Emmental Versicherung Arena Langnau im Emmental',
    'La Chaux-de-Fonds': 'Patinoire des Melezes La Chaux-de-Fonds',
    'Lausanne': 'Vaudoise Arena Lausanne',
    'Lugano': 'Pista La Resega Porza',
    'Rapperswil-Jona': 'St. Galler Kantonalbank Arena Rapperswil-Jona',
    'Servette': 'Patinoire des Vernets Geneva',
    'Visp': 'Lonza Arena Visp',
    'Zug': 'General-Guisan-Strasse 4 Zug',
    'Zurich': 'Swiss Life Arena Zurich',
}
teams = sorted(set(df['Heimmannschaft']) | set(df['Auswärtsmannschaft']))
fehlende_teams = sorted(set(teams) - set(arena_suchen))
if fehlende_teams:
    raise ValueError(f'Bitte ergaenze die Hallensuche fuer: {fehlende_teams}')

# Photon-Ergebnisse lokal cachen und fehlende Hallen einzeln abfragen.
koordinaten_datei = Path('csv_data/photon_arena_coordinates.json')
if koordinaten_datei.exists():
    koordinaten = json.loads(koordinaten_datei.read_text(encoding='utf-8'))
else:
    koordinaten = {}
letzte_geocode_anfrage = None
user_agent = 'EishockeyProjekt-Reisezeitanalyse/1.0'
for team in teams:
    if team in koordinaten:
        continue
    if letzte_geocode_anfrage is not None:
        time.sleep(max(0, 1.1 - (time.monotonic() - letzte_geocode_anfrage)))
    parameter = urlencode({'q': arena_suchen[team], 'limit': 5, 'lang': 'en'})
    anfrage = Request(
        f'https://photon.komoot.io/api/?{parameter}',
        headers={'User-Agent': user_agent},
    )
    with urlopen(anfrage, timeout=30) as antwort:
        antwort_json = json.loads(antwort.read().decode('utf-8'))
    letzte_geocode_anfrage = time.monotonic()
    treffer = [
        feature for feature in antwort_json.get('features', [])
        if feature.get('properties', {}).get('countrycode') == 'CH'
    ]
    if not treffer:
        raise ValueError(f'Keine Schweizer OSM-Halle fuer {team} gefunden: {arena_suchen[team]}')
    feature = treffer[0]
    longitude, latitude = feature['geometry']['coordinates']
    eigenschaften = feature.get('properties', {})
    koordinaten[team] = {
        'lat': float(latitude),
        'lon': float(longitude),
        'name': eigenschaften.get('name') or arena_suchen[team],
        'stadt': eigenschaften.get('city', ''),
        'osm_typ': eigenschaften.get('osm_value', ''),
    }
    koordinaten_datei.write_text(
        json.dumps(koordinaten, ensure_ascii=False, indent=2), encoding='utf-8'
    )

print('Gefundene OSM-Standorte (bitte auf korrekte Halle pruefen):')
for team in teams:
    print(f"{team}: {koordinaten[team].get('name') or arena_suchen[team]} ({koordinaten[team].get('stadt', '')}, {koordinaten[team].get('osm_typ', '')})")

ungueltige_koordinaten = [
    team for team in teams
    if not {'lat', 'lon'} <= koordinaten[team].keys()
]
if ungueltige_koordinaten:
    raise ValueError(f'Ungueltige Koordinaten im Cache fuer: {ungueltige_koordinaten}')

# OSRM-Tabellen in kleinen Batches abrufen und gecachte Routen bei gleicher
# Koordinatenliste wiederverwenden. Zeiten sind Sekunden fuer Auto-Routen.
koordinaten_signatur = hashlib.sha256(json.dumps(
    {team: koordinaten[team] for team in teams}, sort_keys=True
).encode('utf-8')).hexdigest()
routen_datei = Path('csv_data/osrm_travel_times.json')
if routen_datei.exists():
    routen_cache = json.loads(routen_datei.read_text(encoding='utf-8'))
else:
    routen_cache = {}
if routen_cache.get('koordinaten_signatur') != koordinaten_signatur:
    routen_sekunden = {}
    coordinate_list = ';'.join(
        f"{koordinaten[team]['lon']},{koordinaten[team]['lat']}" for team in teams
    )
    for start in range(0, len(teams), 5):
        quellen = list(range(start, min(start + 5, len(teams))))
        parameter = urlencode({
            'sources': ';'.join(map(str, quellen)),
            'destinations': 'all',
            'annotations': 'duration',
        })
        url = f'https://router.project-osrm.org/table/v1/driving/{coordinate_list}?{parameter}'
        anfrage = Request(url, headers={'User-Agent': user_agent})
        with urlopen(anfrage, timeout=60) as antwort:
            routing = json.loads(antwort.read().decode('utf-8'))
        if routing.get('code') != 'Ok':
            raise RuntimeError(f"OSRM konnte Routen nicht berechnen: {routing}")
        for zeile, quellen_index in zip(routing['durations'], quellen):
            if len(zeile) != len(teams) or any(dauer is None for dauer in zeile):
                raise RuntimeError(
                    'OSRM lieferte fuer mindestens ein Hallenpaar keine Auto-Route.'
                )
            routen_sekunden[teams[quellen_index]] = dict(zip(teams, zeile))
    routen_cache = {
        'koordinaten_signatur': koordinaten_signatur,
        'routen_sekunden': routen_sekunden,
    }
    routen_datei.write_text(
        json.dumps(routen_cache, ensure_ascii=False, indent=2), encoding='utf-8'
    )
routen_sekunden = routen_cache['routen_sekunden']
if koordinaten.get('Ajoie', {}).get('osm_typ') == 'bus_stop':
    print('HINWEIS: Ajoie wurde ueber die OSM-Haltestelle Patinoire angenaehert; Koordinaten vor Interpretation pruefen.')

# Fuer ein Auswaertsteam ist der Weg von seiner Halle zur Heimarena massgeblich.
df['Auswaertssieg'] = (df['Auswärts Score'] > df['Heim Score']).astype(int)
df['Reisedauer_min'] = [
    routen_sekunden[auswaerts][heim] / 60
    for heim, auswaerts in zip(df['Heimmannschaft'], df['Auswärtsmannschaft'])
]
if df['Reisedauer_min'].isna().any():
    raise RuntimeError('Mindestens einem Spiel fehlt trotz Routenberechnung eine Reisezeit.')
print(f'\nReisezeiten berechnet fuer alle {len(df)} Spiele.')
print('Fahrzeit bedeutet geschaetzte Auto-Fahrzeit ohne matchtagsbezogene Verkehrslage.')
print('Kartendaten © OpenStreetMap contributors; Geocoding ueber Photon, Routen ueber OSRM.')

# Siegquote nach Fahrzeit-Kategorie, getrennt nach Saison und gesamt.
df['Reise_Kategorie'] = pd.cut(
    df['Reisedauer_min'], bins=[0, 60, 120, np.inf],
    labels=['< 1h', '1-2h', '> 2h'],
)
def berechne_quote(gruppierung):
    ergebnis = gruppierung.agg(
        Spiele=('Auswaertssieg', 'size'),
        Auswaertssiege=('Auswaertssieg', 'sum'),
        Siegquote=('Auswaertssieg', 'mean'),
    ).reset_index()
    ergebnis['Siegquote'] *= 100
    return ergebnis

nach_saison = berechne_quote(df.groupby(['Saison', 'Reise_Kategorie'], observed=False))
gesamt = berechne_quote(df.groupby('Reise_Kategorie', observed=False))
print('\nAuswaertssiegquote nach Fahrzeit und Saison (Prozent):')
print(nach_saison.round(1).to_string(index=False))
print('\nAuswaertssiegquote nach Fahrzeit, beide Saisons (Prozent):')
print(gesamt.round(1).to_string(index=False))

# Logistische Regression auf allen Spielen mit berechneter Reisezeit.
X = sm.add_constant(df['Reisedauer_min'])
modell = sm.Logit(df['Auswaertssieg'], X).fit(disp=0)
print('\nLogistische Regression fuer alle Spiele:')
print(modell.summary())

Gefundene OSM-Standorte (bitte auf korrekte Halle pruefen):
Ajoie: Patinoire (Porrentruy, bus_stop)
Ambri-Piotta: Gottardo Arena (Ambrì, ice_rink)
Bern: PostFinance-Arena (Bern, stadium)
Biel: Tissot Arena (Biel/Bienne, stadium)
Davos: Eisstadion Davos (Davos Platz, sports_hall)
EHC Kloten: SWISS Arena (Kloten, stadium)
Fribourg: BCF Arena (Fribourg, stadium)
La Chaux-de-Fonds: Patinoire (La Chaux-de-Fonds, stadium)
Langnau Tigers: Emmental Versicherung Arena (Langnau im Emmental, sports)
Lausanne: Vaudoise Aréna (Prilly, sports_centre)
Lugano: Pista di ghiaccio della Resega (Porza, ice_rink)
Rapperswil-Jona: St. Galler Kantonalbank Arena (Rapperswil SG, stadium)
Servette: Patinoire des Vernets (Geneva, yes)
Visp: LONZA Arena (Visp, ice_rink)
Zug: General-Guisan-Strasse 4 Zug (Zug, house)
Zurich: Swiss Life Arena (Zurich, fire_detection_system)
HINWEIS: Ajoie wurde ueber die OSM-Haltestelle Patinoire angenaehert; Koordinaten vor Interpretation pruefen.

Reisezeiten berechnet fuer alle 

In [2]:
# Zusammenhang zwischen Fahrzeit und Auswärtstoren prüfen
analyse = df[['Auswärts Score', 'Reisedauer_min', 'Saison']].dropna()

y = analyse['Auswärts Score']
X = pd.get_dummies(
    analyse[['Reisedauer_min', 'Saison']],
    columns=['Saison'],
    drop_first=True,
    dtype=float,
)
X = sm.add_constant(X)

modell_tore = sm.GLM(
    y,
    X,
    family=sm.families.Poisson(),
).fit(cov_type='HC3')

# Effekt einer zusätzlichen Fahrstunde
koeffizient = modell_tore.params['Reisedauer_min']
p_wert = modell_tore.pvalues['Reisedauer_min']
effekt_pro_stunde = np.exp(koeffizient * 60)

print(modell_tore.summary())
print(f'\nVerhältnis der erwarteten Auswärtstore pro zusätzlicher Fahrstunde: {effekt_pro_stunde:.3f}')
print(f'p-Wert für den Zusammenhang mit der Fahrzeit: {p_wert:.4f}')

if p_wert < 0.05:
    print('Der Zusammenhang ist statistisch signifikant (Niveau 5 %).')
else:
    print('Kein statistisch signifikanter Zusammenhang auf dem 5-%-Niveau.')

print('Hinweis: Das Modell zeigt einen Zusammenhang, belegt aber keine Kausalität.')

                 Generalized Linear Model Regression Results                  
Dep. Variable:         Auswärts Score   No. Observations:                 2685
Model:                            GLM   Df Residuals:                     2677
Model Family:                 Poisson   Df Model:                            7
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4949.8
Date:                Tue, 10 Nov 2026   Deviance:                       3037.9
Time:                        16:25:49   Pearson chi2:                 2.67e+03
No. Iterations:                     4   Pseudo R-squ. (CS):           0.004850
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.8912      0.043     20.